# Near-Duplicate Location Diagnosis
Vecchia likelihood Cholesky failure 원인 진단  
특정 날짜에 near-duplicate locations이 있는지 확인합니다.

**배경**: `torch.linalg.cholesky` 실패 → `RuntimeError: element 0 of tensors does not require grad`  
**가설**: 특정 날짜에 동일/근접 좌표 관측치가 있어 공분산 행렬이 near-singular

## 1. Imports

In [ ]:
import sys
gems_tco_path = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
sys.path.append(gems_tco_path)

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from typing import List
from pathlib import Path

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed

## 2. Load Data (gpu_vecc_fit_dynamic_grid 동일 설정)

In [ ]:
space: List[str] = ['1', '1']
lat_lon_resolution = [int(s) for s in space]
mm_cond_number: int = 100
years = ['2024']
month_range = [7]

data_load_instance = load_data_dynamic_processed(config.mac_data_load_path)

lat_range_input = [-3, 2]
lon_range_input = [121, 131]

df_map, ord_mm, nns_map, monthly_mean = data_load_instance.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=lat_lon_resolution,
    mm_cond_number=mm_cond_number,
    years_=years,
    months_=month_range,
    lat_range=lat_range_input,
    lon_range=lon_range_input,
    is_whittle=False
)

In [ ]:
daily_hourly_maps_vecc = []

for day_index in range(31):
    hour_start_index = day_index * 8
    hour_end_index   = (day_index + 1) * 8
    hour_indices     = [hour_start_index, hour_end_index]

    day_hourly_map, _ = data_load_instance.load_working_data(
        df_map,
        monthly_mean,
        hour_indices,
        ord_mm=ord_mm,
        dtype=torch.float64,
        keep_ori=True
    )
    daily_hourly_maps_vecc.append(day_hourly_map)

print(f"Loaded {len(daily_hourly_maps_vecc)} days")

## 3. Diagnosis Functions

In [ ]:
from scipy.spatial import cKDTree

def get_valid_locs(day_map):
    """슬롯별 유효 관측 좌표(lat, lon)를 반환합니다."""
    slot_locs = {}
    for key in sorted(day_map.keys()):
        t_data = day_map[key]
        valid  = ~torch.isnan(t_data[:, 0])
        locs   = t_data[valid, :2].cpu().numpy()   # (n_valid, 2) lat/lon
        slot_locs[key] = locs
    return slot_locs


def nn_min_dist_kdtree(locs, k=2):
    """
    KD-tree로 각 점의 nearest-neighbor 거리를 반환합니다.
    O(N log N), 메모리 O(N).
    k=2: 자기 자신(거리 0)을 제외한 가장 가까운 이웃.
    """
    tree   = cKDTree(locs)
    dists, _ = tree.query(locs, k=k)   # (N, k)
    return dists[:, 1]                 # nearest neighbor (k=2 → index 1)


def cross_nn_min_dist_kdtree(locs_query, locs_ref):
    """
    locs_query의 각 점에서 locs_ref 안의 nearest-neighbor 거리를 반환합니다.
    cross-slot 비교용. O(N log M).
    """
    tree  = cKDTree(locs_ref)
    dists, _ = tree.query(locs_query, k=1)
    return dists   # (N_query,)


def diagnose_near_duplicates(daily_hourly_maps_vecc, day_idx, threshold_deg=0.01):
    """
    특정 날짜의 near-duplicate locations 진단 (KD-tree 기반, O(N log N)).

    Parameters
    ----------
    threshold_deg : float
        이 거리(도) 미만이면 near-duplicate 판정.
        0.044° = GEMS lat 해상도, 0.063° = GEMS lon 해상도
        0.01°  ≈ 1.1 km

    Returns
    -------
    summary : dict  (결과 요약)
    slot_locs : dict of per-slot locs
    """
    day_map   = daily_hourly_maps_vecc[day_idx]
    slot_locs = get_valid_locs(day_map)
    slot_keys = list(slot_locs.keys())

    print(f"{'='*60}")
    print(f"Day {day_idx+1} — Near-Duplicate Location Diagnosis")
    print(f"{'='*60}\n")

    # ── [1] 슬롯별 관측 수 ───────────────────────────────────────
    print("[1] Observations per slot")
    for key, locs in slot_locs.items():
        print(f"  slot {key}: {len(locs):6d} obs")
    total_n = sum(len(v) for v in slot_locs.values())
    print(f"  Total  : {total_n}\n")

    # ── [2] 슬롯 내 near-duplicate (KD-tree) ─────────────────────
    print("[2] Within-slot near-duplicates  (KD-tree)")
    within_min = {}
    any_within = False
    for key, locs in slot_locs.items():
        if len(locs) < 2:
            print(f"  slot {key}: only {len(locs)} obs — skip")
            continue
        nn_dist = nn_min_dist_kdtree(locs)
        min_d   = nn_dist.min()
        n_dup   = int((nn_dist < threshold_deg).sum())
        within_min[key] = min_d
        if n_dup > 0:
            any_within = True
            # 실제 pair 좌표 찾기 (진단용 — threshold 이하인 소수만 full search)
            bad_idx = np.where(nn_dist < threshold_deg)[0]
            tree    = cKDTree(locs)
            print(f"  slot {key}: *** {n_dup} obs with NN < {threshold_deg}° ***")
            for i in bad_idx[:5]:
                d, j = tree.query(locs[i], k=2)
                j = j[1]   # nearest neighbor index
                print(f"    ({locs[i,0]:.5f},{locs[i,1]:.5f}) ↔ "
                      f"({locs[j,0]:.5f},{locs[j,1]:.5f})  dist={d[1]:.6f}°")
        else:
            print(f"  slot {key}: min_nn_dist={min_d:.6f}°  OK")
    if not any_within:
        print("  → No within-slot near-duplicates\n")

    # ── [3] Cross-slot near-duplicate (KD-tree) ──────────────────
    print("[3] Cross-slot near-duplicates  (KD-tree, each slot vs others)")
    cross_results = {}
    any_cross = False
    for i, key_q in enumerate(slot_keys):
        locs_q = slot_locs[key_q]
        # 다른 모든 슬롯을 합쳐서 reference로 사용
        other_locs = np.vstack([slot_locs[k] for k in slot_keys if k != key_q])
        nn_cross   = cross_nn_min_dist_kdtree(locs_q, other_locs)
        min_c      = nn_cross.min()
        n_dup_c    = int((nn_cross < threshold_deg).sum())
        cross_results[key_q] = {'min': min_c, 'n_dup': n_dup_c, 'nn': nn_cross}
        if n_dup_c > 0:
            any_cross = True
            print(f"  slot {key_q}: *** {n_dup_c} obs with cross-slot NN < {threshold_deg}° ***  "
                  f"(min={min_c:.6f}°)")
        else:
            print(f"  slot {key_q}: min_cross_NN={min_c:.6f}°  OK")
    if not any_cross:
        print("  → No cross-slot near-duplicates\n")

    # ── [4] 전체 합산 NN 거리 분포 ───────────────────────────────
    print("\n[4] Stacked NN-distance distribution (within + cross, KD-tree)")
    all_locs   = np.vstack(list(slot_locs.values()))
    nn_all     = nn_min_dist_kdtree(all_locs)   # O(N log N), no N² matrix
    n_dup_all  = int((nn_all < threshold_deg).sum())
    print(f"  Total stacked obs : {len(all_locs)}")
    print(f"  Obs with NN < {threshold_deg}°: {n_dup_all}")
    for thr in [0.001, 0.005, 0.010, 0.044, 0.063]:
        cnt = int((nn_all < thr).sum())
        print(f"  NN dist < {thr:.3f}° : {cnt:5d} obs")
    print(f"  Overall min    : {nn_all.min():.6f}°")
    print(f"  Overall mean   : {nn_all.mean():.6f}°")
    print(f"  Overall median : {np.median(nn_all):.6f}°")

    summary = {
        'day': day_idx + 1,
        'total_obs': total_n,
        'within_any': any_within,
        'cross_any':  any_cross,
        'stacked_min_nn': nn_all.min(),
        'n_dup_stacked': n_dup_all,
        'nn_all': nn_all,
        'all_locs': all_locs,
    }
    return summary, slot_locs

## 4. Run Diagnosis — Specific Problem Days

In [ ]:
# ── 에러가 발생한 날짜를 여기에 입력 (0-based index) ──
PROBLEM_DAYS  = [26]       # 예: day 27 = index 26
THRESHOLD_DEG = 0.01       # near-duplicate 판정 기준 (도)

results = {}
for day_idx in PROBLEM_DAYS:
    summary, slot_locs = diagnose_near_duplicates(
        daily_hourly_maps_vecc, day_idx, threshold_deg=THRESHOLD_DEG
    )
    results[day_idx] = (summary, slot_locs)
    print()

## 5. Scan All 31 Days — Min Distance Summary

In [ ]:
# 전체 31일 스캔 — KD-tree 기반, 빠름 O(N log N) per day
import time

scan_summary = []
for day_idx in range(31):
    t0        = time.time()
    slot_locs = get_valid_locs(daily_hourly_maps_vecc[day_idx])
    all_locs  = np.vstack(list(slot_locs.values()))
    nn_all    = nn_min_dist_kdtree(all_locs)
    n_dup     = int((nn_all < THRESHOLD_DEG).sum())
    elapsed   = time.time() - t0
    scan_summary.append({
        'day':          day_idx + 1,
        'total_obs':    len(all_locs),
        'min_nn_dist':  nn_all.min(),
        'mean_nn_dist': nn_all.mean(),
        'n_near_dup':   n_dup,
    })
    flag = '  *** NEAR-DUP ***' if n_dup > 0 else ''
    print(f"Day {day_idx+1:2d}: n={len(all_locs):6d}  "
          f"min_nn={nn_all.min():.5f}°  "
          f"near_dup={n_dup:4d}  ({elapsed:.1f}s){flag}")

## 6. Visualize — Min Distance by Day

In [ ]:
days      = [s['day']        for s in scan_summary]
min_dists = [s['min_nn_dist'] for s in scan_summary]
n_dups    = [s['n_near_dup']  for s in scan_summary]

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
fig.suptitle('Near-Duplicate Location Scan — July 2024 (31 days)', fontsize=12, fontweight='bold')

# (a) 날짜별 최소 NN 거리
ax = axes[0]
colors = ['tomato' if d > 0 else 'steelblue' for d in n_dups]
ax.bar(days, min_dists, color=colors, edgecolor='black', linewidth=0.4)
ax.axhline(THRESHOLD_DEG, ls='--', color='red', lw=1.2, label=f'threshold={THRESHOLD_DEG}°')
ax.axhline(0.044, ls=':', color='gray', lw=1.0, label='GEMS lat res (0.044°)')
ax.set_xlabel('Day'); ax.set_ylabel('Min NN distance (°)')
ax.set_title('(a) Min nearest-neighbor distance across all slots  [red = near-duplicate found]')
ax.set_xticks(days); ax.legend(fontsize=8)

# (b) 날짜별 near-duplicate obs 수
ax = axes[1]
ax.bar(days, n_dups, color='tomato', edgecolor='black', linewidth=0.4)
ax.set_xlabel('Day'); ax.set_ylabel(f'# obs with NN < {THRESHOLD_DEG}°')
ax.set_title(f'(b) Number of near-duplicate observations (NN dist < {THRESHOLD_DEG}°)')
ax.set_xticks(days)

plt.tight_layout()
plt.show()

## 7. Visualize — Spatial Map of Near-Duplicate Locations (specific day)

In [ ]:
# 문제 날짜의 공간 분포 시각화
DAY_IDX = PROBLEM_DAYS[0]

summary, slot_locs = results[DAY_IDX]
all_locs = summary['all_locs']
nn_all   = summary['nn_all']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Day {DAY_IDX+1} — Observation Locations', fontsize=12, fontweight='bold')

# (a) 슬롯별 색상으로 전체 분포
ax = axes[0]
cmap = plt.cm.get_cmap('tab10', len(slot_locs))
for k, (key, locs) in enumerate(slot_locs.items()):
    ax.scatter(locs[:, 1], locs[:, 0], s=2, alpha=0.4,
               color=cmap(k), label=key.split('_')[-1])
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('(a) All slots (color per slot)')
ax.legend(fontsize=7, markerscale=3)

# (b) NN 거리로 색칠 (가까울수록 빨간색)
ax = axes[1]
vmin = max(nn_all.min(), 1e-6)
sc   = ax.scatter(all_locs[:, 1], all_locs[:, 0], s=2,
                  c=nn_all, cmap='RdYlGn',
                  norm=mcolors.LogNorm(vmin=vmin, vmax=nn_all.max()))
plt.colorbar(sc, ax=ax, label='NN distance (°, log scale)')

# near-duplicate obs 강조
bad_mask = nn_all < THRESHOLD_DEG
if bad_mask.sum() > 0:
    ax.scatter(all_locs[bad_mask, 1], all_locs[bad_mask, 0],
               s=20, c='red', marker='x', zorder=5,
               label=f'NN<{THRESHOLD_DEG}° ({bad_mask.sum()} obs)')
    ax.legend(fontsize=8)

ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'(b) NN distance  [red x = NN < {THRESHOLD_DEG}°]')

plt.tight_layout()
plt.show()
print(f"Near-duplicate obs: {bad_mask.sum()}  (NN < {THRESHOLD_DEG}°)")
print(f"Min NN dist: {nn_all.min():.6f}°  |  Mean: {nn_all.mean():.6f}°")